# 05 · uv + Wheel

## 왜 wheel 인가

`code_paths` 는 단순하지만 **평탄화 / relative import / transitive deps** 문제가 있습니다 (→ `04_code_paths`).
프로덕션 권장은:

> **`uv build` 로 wheel 만들기 → `code_paths=[wheel]` + `extra_pip_requirements=["code/<wheel>"]` 로 모델에 번들**

## 정석 패턴

```python
mlflow.pyfunc.log_model(
    name="model",
    python_model=MyModel(),
    code_paths=["dist/churn_preproc-0.1.0-py3-none-any.whl"],
    extra_pip_requirements=["code/churn_preproc-0.1.0-py3-none-any.whl"],  # ← code/ 접두사 필수
)
```

- `code_paths` 가 wheel 을 `<model>/code/` 로 복사
- `extra_pip_requirements` 의 `code/<wheel>` 은 unpacked 모델 root 기준 상대경로 — 서빙 빌더가 pip install

## 작동 X 패턴 (흔한 실수)

```python
mlflow.pyfunc.log_model(
    artifacts={"wheel": "./mypkg.whl"},
    pip_requirements=["./artifacts/mypkg.whl"],   # ❌ pip resolver 가 artifacts 안 봄
)
```

`artifacts` 는 `context.artifacts["wheel"]` 로 **predict 안에서 접근**하는 용도.
pip resolver 가 검사하지 않으므로 dep 해결 실패.

## Step 1. uv 설치 + 패키지 빌드

`src/churn_preproc/` 디렉토리에 이미 패키지 소스가 있습니다 (Repo / Workspace 동기화 시).
없다면 README 의 "Setup" 섹션을 따라 패키지를 먼저 생성하세요.

In [ ]:
%sh
# uv 설치 (없는 경우만)
pip install -q "uv>=0.5.0"
uv --version

### Wheel 빌드

디렉토리 찾았으면 거기서 `uv build` 실행. wheel 은 `dist/` 에 생성됩니다.

In [ ]:
import subprocess, os, glob

# 노트북 위치 기준: 같은 디렉토리에 churn_preproc/ 패키지 소스가 함께 제공됨.
# Databricks 노트북 컨텍스트에서 현재 노트북의 워크스페이스 경로를 가져옴.
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
notebook_dir = "/Workspace" + os.path.dirname(ctx.notebookPath().get())
PKG_DIR = f"{notebook_dir}/churn_preproc"

assert os.path.exists(f"{PKG_DIR}/pyproject.toml"), (
    f"pyproject.toml not found at {PKG_DIR}. "
    f"churn_preproc 패키지가 노트북과 같은 디렉토리에 sync 되어 있어야 합니다."
)
print(f"PKG_DIR = {PKG_DIR}")

# uv build → /tmp/wheels
result = subprocess.run(
    ["uv", "build", "--out-dir", "/tmp/wheels"],
    cwd=PKG_DIR, capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("uv build 실패")

WHEEL = glob.glob("/tmp/wheels/churn_preproc-*.whl")[0]
print(f"\n✓ Built: {WHEEL}")


## Step 2. 빌드한 wheel 로컬 테스트

log_model 전에 wheel 이 설치되고 import 가 잘 되는지 검증.

In [ ]:
%pip install -q /tmp/wheels/churn_preproc-0.1.0-py3-none-any.whl
%restart_python

In [ ]:
%run ./config

In [ ]:
import mlflow, os, pandas as pd, joblib, glob, sklearn, numpy as np, cloudpickle
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(experiment_path)

# wheel 사용 확인
from churn_preproc import preprocess, __version__
print(f"churn_preproc=={__version__}")

# %restart_python 이후 변수 손실 — wheel 경로 재정의
WHEEL = glob.glob("/tmp/wheels/churn_preproc-*.whl")[0]
print(f"WHEEL = {WHEEL}")


## Step 3. 학습 — wheel 의 함수 사용

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

pdf = spark.table(f"{catalog}.{schema}.customers").toPandas()
pdf = preprocess(pdf)   # ← wheel 의 preprocess

NUMERIC = ["age", "tenure_months", "monthly_charges", "total_charges", "support_tickets", "ltv"]
CATEGORICAL = ["risk_bucket", "tenure_band"]
FEATURES = NUMERIC + CATEGORICAL

X_train, X_test, y_train, y_test = train_test_split(
    pdf[FEATURES], pdf["churned"], test_size=0.2, random_state=42
)

pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ], remainder="passthrough")),
    ("rf", RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)),
])
pipe.fit(X_train, y_train)
print(f"test accuracy = {pipe.score(X_test, y_test):.3f}")

## Step 4. PythonModel — wheel 에서 import

In [ ]:
class ChurnWithWheel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        import joblib
        self.model = joblib.load(context.artifacts["model"])

    def predict(self, context, model_input: pd.DataFrame, params=None):
        # ← 서빙 컨테이너 안에서 wheel 이 설치되어 있음
        from churn_preproc import preprocess
        enriched = preprocess(model_input)
        # 학습 시 사용한 컬럼 순서 유지
        cols = ["age", "tenure_months", "monthly_charges", "total_charges",
                "support_tickets", "ltv", "risk_bucket", "tenure_band"]
        return pd.DataFrame({"prediction": self.model.predict(enriched[cols])})

## Step 5. 모델을 Volume 에 저장 + log_model (정석 패턴)

In [ ]:
artifacts_dir = f"{volume_path}/05_wheel"
os.makedirs(artifacts_dir, exist_ok=True)
model_path = f"{artifacts_dir}/pipeline.joblib"
joblib.dump(pipe, model_path)

# input_example 은 raw input — preprocess 가 컬럼 추가
raw_example = pdf[["age", "tenure_months", "monthly_charges", "total_charges", "support_tickets"]].iloc[:5]

from mlflow.models import infer_signature
sig = infer_signature(raw_example, pd.DataFrame({"prediction": [0]*5}))

wheel_basename = os.path.basename(WHEEL)
print(f"wheel = {wheel_basename}")

with mlflow.start_run(run_name="wheel_packaging"):
    info = mlflow.pyfunc.log_model(
        name="model",
        python_model=ChurnWithWheel(),
        artifacts={"model": model_path},
        code_paths=[WHEEL],                                  # ← wheel 을 code/ 로 복사
        signature=sig,
        input_example=raw_example,
        registered_model_name=model_wheel,
        pip_requirements=[                                   # ← lockdown: 자동 추론 churn-preproc==0.1.0 (PyPI 미공개) 제외
            f"mlflow=={mlflow.__version__}",                 #   학습 시점 환경 버전을 그대로 pin → endpoint unpickle 호환
            f"cloudpickle=={cloudpickle.__version__}",
            f"scikit-learn=={sklearn.__version__}",
            f"pandas=={pd.__version__}",
            f"numpy=={np.__version__}",
            f"joblib=={joblib.__version__}",
            f"code/{wheel_basename}",                        # ← endpoint 환경에서 wheel 직접 pip install
        ],
    )

print(f"✓ {model_wheel} v{info.registered_model_version}")

## Step 6. 모델 디렉토리 확인 — wheel 이 code/ 안에 들어갔는지

In [ ]:
local = mlflow.artifacts.download_artifacts(info.model_uri)
print("=== requirements.txt ===")
print(open(os.path.join(local, "requirements.txt")).read())

print("=== code/ 내용 ===")
for f in os.listdir(os.path.join(local, "code")):
    print(f"  {f}")

## Step 7. uv 격리 환경에서 검증

`mlflow.models.predict(env_manager="uv")` 가 wheel 까지 포함한 새 환경을 만들어 실행합니다.
**서빙 빌드와 동일한 의존성 해결 경로**를 거치므로 production 사고를 사전에 잡아줍니다.

In [ ]:
mlflow.models.predict(
    model_uri=info.model_uri,
    input_data={"dataframe_split": {
        "columns": list(raw_example.columns),
        "data":    raw_example.values.tolist(),
    }},
    env_manager="uv",
)

## Step 8. Alias 설정

In [ ]:
from mlflow import MlflowClient
MlflowClient().set_registered_model_alias(model_wheel, "Champion", info.registered_model_version)
print(f"✓ {model_wheel}@Champion → v{info.registered_model_version}")

## 정리 — code_paths vs wheel

| 측면 | `code_paths=["mypkg/"]` (raw) | `code_paths=["dist/...whl"]` (wheel) |
| --- | --- | --- |
| 빌드 스텝 | 없음 | `uv build` |
| 중첩 패키지 | 평탄화로 **깨짐** | 정상 pip install |
| relative import | 자주 깨짐 | 정상 |
| transitive deps | 수동 listing | pyproject.toml 에서 자동 |
| 버전 관리 | 없음 | semver in filename |
| 프로덕션 | 비권장 | **권장** |

### 함정들

- **버전 미증가 시 캐싱**: wheel 은 스냅샷. 매 반복마다 `0.1.0 → 0.1.1` 로 bump 해야 pip cache 가 새 wheel 사용.
- **Python ABI 태그**: `py3-none-any` (pure Python) 면 어디서나 OK. C-extension 이 있으면 `manylinux2014_x86_64` wheel 필요.
- **DBR runtime 호환성**: `pyproject.toml` 의 `requires-python` 을 DBR ML runtime 의 Python 버전과 맞추세요 (DBR ML 15.x = 3.11).
- **UC Volume 사용 시**: endpoint service principal 에 Volume READ grant 필요.
- **code_paths 디렉토리 + wheel 혼합 X**: import shadowing 발생 — wheel 만 쓰세요.

→ 다음: **`06_model_serving`** — endpoint 생성, 호출, blue/green